# Understanding Linkage vs Linkage Disequilibrium (LD)
## An Interactive Visual Tutorial

**Learning Objectives:**
- Distinguish between linkage and linkage disequilibrium
- Visualize how LD changes over generations
- Understand when and why LD persists or decays
- Apply concepts to real-world genetics scenarios

---

In [ ]:
# Install and import required packages
!pip install ipywidgets matplotlib numpy pandas seaborn plotly -q

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, widgets, HBox, VBox
from IPython.display import display, HTML, Markdown
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ All packages loaded successfully!")

---
## Part 1: The Key Difference - Visual Summary

Let's start with the core distinction:

In [ ]:
def show_key_difference():
    """
    Visual comparison of linkage vs LD
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # LEFT: LINKAGE
    ax1.set_xlim(0, 10)
    ax1.set_ylim(0, 10)
    ax1.axis('off')
    ax1.set_title('LINKAGE\n(Physical Proximity)', fontsize=18, fontweight='bold', color='#2E86AB')
    
    # Draw chromosome
    chrom_y = 5
    ax1.plot([1, 9], [chrom_y, chrom_y], 'k-', linewidth=12, alpha=0.6)
    
    # Gene positions
    ax1.plot(2, chrom_y, 'ro', markersize=30, label='Gene A')
    ax1.plot(4, chrom_y, 'bo', markersize=30, label='Gene B')
    ax1.text(2, chrom_y+1.2, 'Gene A', ha='center', fontsize=14, fontweight='bold')
    ax1.text(4, chrom_y+1.2, 'Gene B', ha='center', fontsize=14, fontweight='bold')
    
    # Distance annotation
    ax1.annotate('', xy=(4, chrom_y-1), xytext=(2, chrom_y-1),
                arrowprops=dict(arrowstyle='<->', color='green', lw=3))
    ax1.text(3, chrom_y-1.5, '10 cM', ha='center', fontsize=14, 
            fontweight='bold', color='green',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
    
    # Key points
    info_text = [
        '✓ About LOCATION',
        '✓ Fixed in genome',
        '✓ Same for everyone',
        '✓ Never changes',
        '✓ Measured in cM'
    ]
    for i, text in enumerate(info_text):
        ax1.text(0.5, 8.5-i*0.8, text, fontsize=11, family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    
    # RIGHT: LINKAGE DISEQUILIBRIUM
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 10)
    ax2.axis('off')
    ax2.set_title('LINKAGE DISEQUILIBRIUM\n(Allele Association)', fontsize=18, fontweight='bold', color='#A23B72')
    
    # Draw population representation
    # Haplotype frequencies
    haplotypes = ['A₁-B₁', 'A₁-B₂', 'A₂-B₁', 'A₂-B₂']
    frequencies = [0.49, 0.01, 0.01, 0.49]  # Strong LD
    colors_haplo = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    
    # Create pie chart representation
    center_x, center_y = 5, 6
    radius = 2
    start_angle = 0
    
    for i, (haplo, freq, color) in enumerate(zip(haplotypes, frequencies, colors_haplo)):
        angle = 360 * freq
        wedge = plt.Circle((center_x, center_y), radius, color=color, alpha=0.7)
        theta1 = start_angle
        theta2 = start_angle + angle
        
        # Draw wedge
        theta = np.linspace(np.radians(theta1), np.radians(theta2), 100)
        x = center_x + radius * np.cos(theta)
        y = center_y + radius * np.sin(theta)
        vertices = [(center_x, center_y)] + list(zip(x, y))
        poly = plt.Polygon(vertices, color=color, alpha=0.7, edgecolor='black', linewidth=2)
        ax2.add_patch(poly)
        
        # Label
        mid_angle = np.radians((theta1 + theta2) / 2)
        label_x = center_x + (radius + 0.8) * np.cos(mid_angle)
        label_y = center_y + (radius + 0.8) * np.sin(mid_angle)
        if freq > 0.1:
            ax2.text(label_x, label_y, f'{haplo}\n{freq:.0%}', 
                    ha='center', va='center', fontsize=10, fontweight='bold')
        
        start_angle += angle
    
    ax2.text(center_x, center_y, 'Population\nHaplotypes', ha='center', va='center',
            fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Key points
    info_text2 = [
        '✓ About ASSOCIATION',
        '✓ Changes over time',
        '✓ Population-specific',
        '✓ Decays with recombination',
        '✓ Measured in D, r²'
    ]
    for i, text in enumerate(info_text2):
        ax2.text(0.5, 3-i*0.6, text, fontsize=11, family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))
    
    plt.suptitle('LINKAGE vs LINKAGE DISEQUILIBRIUM', 
                fontsize=20, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "="*80)
    print("KEY DISTINCTION")
    print("="*80)
    print("\n🧬 LINKAGE:      'How close are genes on the chromosome?'")
    print("📊 LD:           'How often do specific alleles occur together in a population?'")
    print("\n" + "="*80)

# Display the comparison
show_key_difference()

---
## Part 2: Interactive Simulation - LD Decay Over Generations

Watch how linkage disequilibrium changes over generations!

In [ ]:
def calculate_ld_over_time(initial_ld, recombination_rate, generations):
    """
    Calculate LD decay over generations
    LD(t) = LD(0) * (1 - r)^t
    """
    ld_values = []
    for gen in range(generations + 1):
        ld = initial_ld * ((1 - recombination_rate) ** gen)
        ld_values.append(ld)
    return ld_values

def visualize_ld_decay(map_distance_cM, initial_ld_percent):
    """
    Visualize how LD decays based on linkage
    """
    # Convert map distance to recombination frequency
    recombination_rate = map_distance_cM / 100  # cM to proportion
    initial_ld = initial_ld_percent / 100
    
    generations = 50
    
    # Calculate LD decay
    ld_values = calculate_ld_over_time(initial_ld, recombination_rate, generations)
    
    # Also calculate for comparison scenarios
    ld_unlinked = calculate_ld_over_time(initial_ld, 0.5, generations)  # Unlinked (50 cM)
    ld_tight = calculate_ld_over_time(initial_ld, 0.01, generations)     # Tightly linked (1 cM)
    
    # Create figure
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    # 1. Main LD decay plot
    ax1 = fig.add_subplot(gs[0:2, :])
    generations_list = list(range(generations + 1))
    
    # Plot current scenario
    ax1.plot(generations_list, ld_values, 'o-', linewidth=3, markersize=8,
            color='#2E86AB', label=f'Your Scenario ({map_distance_cM} cM)', alpha=0.8)
    
    # Plot comparison scenarios
    ax1.plot(generations_list, ld_unlinked, '--', linewidth=2, 
            color='#A23B72', label='Unlinked genes (50 cM)', alpha=0.6)
    ax1.plot(generations_list, ld_tight, '-.', linewidth=2,
            color='#06A77D', label='Tightly linked (1 cM)', alpha=0.6)
    
    ax1.set_xlabel('Generations', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Linkage Disequilibrium (D\')', fontsize=14, fontweight='bold')
    ax1.set_title(f'LD Decay Over Generations\nMap Distance: {map_distance_cM} cM | Recombination Rate: {recombination_rate:.2%}',
                 fontsize=16, fontweight='bold')
    ax1.legend(fontsize=12, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-0.05, max(initial_ld, 0.1) + 0.1)
    
    # Add shading to show decay rate
    ax1.fill_between(generations_list, 0, ld_values, alpha=0.2, color='#2E86AB')
    
    # Add generation markers
    key_generations = [1, 10, 25, 50]
    for gen in key_generations:
        if gen <= generations:
            ax1.axvline(gen, color='gray', linestyle=':', alpha=0.5)
            ax1.text(gen, ax1.get_ylim()[1]*0.95, f'Gen {gen}',
                    ha='center', fontsize=9, rotation=90, alpha=0.7)
    
    # 2. Half-life calculation
    ax2 = fig.add_subplot(gs[2, 0])
    ax2.axis('off')
    
    # Calculate half-life (generations for LD to reach 50% of initial)
    half_ld = initial_ld / 2
    half_life = None
    for i, ld in enumerate(ld_values):
        if ld <= half_ld:
            half_life = i
            break
    
    # Calculate time to near-zero (5% of initial)
    near_zero_threshold = initial_ld * 0.05
    time_to_zero = None
    for i, ld in enumerate(ld_values):
        if ld <= near_zero_threshold:
            time_to_zero = i
            break
    
    stats_text = f"""
    DECAY STATISTICS:
    
    Initial LD:        {initial_ld_percent:.1f}%
    Map Distance:      {map_distance_cM:.1f} cM
    Recomb. Rate:      {recombination_rate:.2%}
    
    LD after 1 gen:    {ld_values[1]:.4f}
    LD after 10 gen:   {ld_values[10]:.4f}
    LD after 50 gen:   {ld_values[50]:.4f}
    
    Half-life:         {half_life if half_life else '>50'} generations
    Nearly gone (5%):  {time_to_zero if time_to_zero else '>50'} generations
    """
    
    ax2.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    # 3. Interpretation
    ax3 = fig.add_subplot(gs[2, 1])
    ax3.axis('off')
    
    # Determine interpretation based on map distance
    if map_distance_cM < 5:
        interpretation = (
            "VERY TIGHT LINKAGE\n\n"
            "✓ LD persists for MANY generations\n"
            "✓ Genes stay together\n"
            "✓ Excellent for gene mapping\n"
            "✓ Useful genetic marker\n\n"
            "Example: HLA genes in humans"
        )
        interp_color = 'lightgreen'
    elif map_distance_cM < 20:
        interpretation = (
            "MODERATE LINKAGE\n\n"
            "✓ LD decays moderately\n"
            "✓ Some recombination occurs\n"
            "✓ Good for mapping studies\n"
            "✓ Population-dependent\n\n"
            "Example: Most linked genes"
        )
        interp_color = 'lightyellow'
    elif map_distance_cM < 50:
        interpretation = (
            "LOOSE LINKAGE\n\n"
            "✓ LD decays rapidly\n"
            "✓ Frequent recombination\n"
            "✓ Harder to detect in populations\n"
            "✓ Historical info only\n\n"
            "Example: Distant markers"
        )
        interp_color = 'peachpuff'
    else:
        interpretation = (
            "INDEPENDENT ASSORTMENT\n\n"
            "✓ LD disappears in 1-2 generations\n"
            "✓ No physical linkage\n"
            "✓ Random association only\n"
            "✓ Not useful for mapping\n\n"
            "Example: Different chromosomes"
        )
        interp_color = 'lightcoral'
    
    ax3.text(0.5, 0.5, interpretation, fontsize=10, 
            verticalalignment='center', horizontalalignment='center',
            bbox=dict(boxstyle='round', facecolor=interp_color, alpha=0.6))
    
    plt.tight_layout()
    plt.show()
    
    # Print additional insights
    print("\n" + "="*80)
    print("KEY INSIGHTS:")
    print("="*80)
    print(f"\n1. With {map_distance_cM} cM distance:")
    print(f"   - Each generation, LD retains {(1-recombination_rate)*100:.1f}% of its value")
    print(f"   - After 10 generations: {(ld_values[10]/initial_ld)*100:.1f}% remains")
    
    if recombination_rate < 0.1:
        print("\n2. This represents STRONG linkage")
        print("   - LD will be detectable for many generations")
        print("   - Great for gene mapping and association studies")
    elif recombination_rate < 0.3:
        print("\n2. This represents MODERATE linkage")
        print("   - LD decays but persists for several generations")
        print("   - Useful for population genetics studies")
    else:
        print("\n2. This represents WEAK or NO linkage")
        print("   - LD disappears quickly")
        print("   - Genes behave nearly independently")

# Interactive widget
print("\n📊 INTERACTIVE LD DECAY SIMULATOR")
print("Adjust the sliders to see how linkage affects LD persistence\n")

interact(visualize_ld_decay,
         map_distance_cM=FloatSlider(min=0.1, max=100, step=0.5, value=10,
                                     description='Map Distance (cM):',
                                     style={'description_width': 'initial'}),
         initial_ld_percent=FloatSlider(min=10, max=100, step=5, value=100,
                                       description='Initial LD (%):',
                                       style={'description_width': 'initial'}));

---
## Part 3: Four Key Scenarios

Let's explore the four possible combinations of linkage and LD:

In [ ]:
def show_four_scenarios():
    """
    Visualize all four scenarios: Linkage +/- and LD +/-
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Four Possible Scenarios: Linkage × LD', fontsize=18, fontweight='bold')
    
    scenarios = [
        {
            'title': 'Scenario 1: LINKAGE + LD\n(Newly formed population)',
            'linkage': True,
            'ld': True,
            'map_dist': 5,
            'haplotypes': [0.45, 0.05, 0.05, 0.45],
            'description': 'Genes are close + Non-random association\n\nMost common in:\n• New populations\n• Recent mutations\n• Strong selection\n• Population bottleneck',
            'color': 'lightgreen'
        },
        {
            'title': 'Scenario 2: LINKAGE but NO LD\n(Old, mixed population)',
            'linkage': True,
            'ld': False,
            'map_dist': 5,
            'haplotypes': [0.25, 0.25, 0.25, 0.25],
            'description': 'Genes are close + Random association\n\nOccurs in:\n• Old populations\n• After many generations\n• Random mating\n• No selection',
            'color': 'lightyellow'
        },
        {
            'title': 'Scenario 3: NO LINKAGE but LD\n(Population admixture)',
            'linkage': False,
            'ld': True,
            'map_dist': 50,
            'haplotypes': [0.40, 0.10, 0.10, 0.40],
            'description': 'Genes far/different chr + Non-random\n\nCauses:\n• Population mixing\n• Migration\n• Selection on combinations\n• Recent admixture',
            'color': 'lightblue'
        },
        {
            'title': 'Scenario 4: NO LINKAGE, NO LD\n(Equilibrium)',
            'linkage': False,
            'ld': False,
            'map_dist': 50,
            'haplotypes': [0.25, 0.25, 0.25, 0.25],
            'description': 'Genes far/different chr + Random\n\nExpected state:\n• Independent chromosomes\n• After many generations\n• No special forces\n• Equilibrium state',
            'color': 'lightcoral'
        }
    ]
    
    for idx, (ax, scenario) in enumerate(zip(axes.flat, scenarios)):
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 10)
        ax.axis('off')
        ax.set_title(scenario['title'], fontsize=13, fontweight='bold', pad=10)
        
        # Draw chromosome representation
        if scenario['linkage']:
            # Linked: genes on same chromosome, close together
            chrom_y = 7
            ax.plot([1, 9], [chrom_y, chrom_y], 'k-', linewidth=8, alpha=0.6)
            ax.plot(2.5, chrom_y, 'ro', markersize=20)
            ax.plot(4, chrom_y, 'bo', markersize=20)
            ax.text(2.5, chrom_y+0.7, 'A', ha='center', fontsize=12, fontweight='bold')
            ax.text(4, chrom_y+0.7, 'B', ha='center', fontsize=12, fontweight='bold')
            ax.annotate('', xy=(4, chrom_y-0.6), xytext=(2.5, chrom_y-0.6),
                       arrowprops=dict(arrowstyle='<->', color='green', lw=2))
            ax.text(3.25, chrom_y-1, f'{scenario["map_dist"]} cM', ha='center',
                   fontsize=9, color='green', fontweight='bold')
        else:
            # Unlinked: genes on different chromosomes
            chrom_y1, chrom_y2 = 7.5, 6.2
            ax.plot([1, 4], [chrom_y1, chrom_y1], 'k-', linewidth=8, alpha=0.6)
            ax.plot([6, 9], [chrom_y2, chrom_y2], 'gray', linewidth=8, alpha=0.6)
            ax.plot(2.5, chrom_y1, 'ro', markersize=20)
            ax.plot(7.5, chrom_y2, 'bo', markersize=20)
            ax.text(2.5, chrom_y1+0.5, 'A', ha='center', fontsize=12, fontweight='bold')
            ax.text(7.5, chrom_y2+0.5, 'B', ha='center', fontsize=12, fontweight='bold')
            ax.text(2.5, chrom_y1-0.7, 'Chr 1', ha='center', fontsize=9, style='italic')
            ax.text(7.5, chrom_y2-0.7, 'Chr 2', ha='center', fontsize=9, style='italic')
        
        # Draw haplotype frequency bars
        haplotypes = ['A₁B₁', 'A₁B₂', 'A₂B₁', 'A₂B₂']
        frequencies = scenario['haplotypes']
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
        
        bar_width = 0.4
        x_positions = [1.5, 3, 4.5, 6]
        y_base = 4.5
        
        for i, (haplo, freq, color, x_pos) in enumerate(zip(haplotypes, frequencies, colors, x_positions)):
            # Draw bar
            height = freq * 6  # Scale for visibility
            rect = plt.Rectangle((x_pos - bar_width/2, y_base - height), 
                                bar_width, height, 
                                color=color, alpha=0.7, edgecolor='black', linewidth=2)
            ax.add_patch(rect)
            
            # Label
            ax.text(x_pos, y_base - height - 0.3, haplo, ha='center', fontsize=9, fontweight='bold')
            ax.text(x_pos, y_base - height/2, f'{freq:.0%}', ha='center', fontsize=10, fontweight='bold')
        
        # Add description
        ax.text(5, 1.5, scenario['description'], ha='center', va='center',
               fontsize=9, bbox=dict(boxstyle='round', facecolor=scenario['color'], alpha=0.5))
        
        # Add LD indicator
        if scenario['ld']:
            ld_text = '✓ LD Present'
            ld_color = 'green'
        else:
            ld_text = '✗ No LD (Equilibrium)'
            ld_color = 'red'
        
        ax.text(5, 0.3, ld_text, ha='center', fontsize=11, fontweight='bold',
               color=ld_color, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print summary table
    print("\n" + "="*80)
    print("SCENARIO SUMMARY")
    print("="*80)
    print("\n| Scenario | Linkage | LD | Common Cause | Stability |")
    print("|----------|---------|----|--------------|-----------| ")
    print("| 1        | Yes     | Yes| New pop      | Temporary |")
    print("| 2        | Yes     | No | Old pop      | Stable    |")
    print("| 3        | No      | Yes| Admixture    | Temporary |")
    print("| 4        | No      | No | Equilibrium  | Stable    |")
    print("\n" + "="*80)

show_four_scenarios()

---
## Part 4: Interactive Population Simulation

Create your own population and watch how LD evolves!

In [ ]:
def simulate_population_ld(map_distance, initial_freq_A1B1, initial_freq_A1B2, generations_to_show):
    """
    Simulate population genetics with visualization of haplotype frequencies over time
    """
    # Initialize haplotype frequencies
    # We set two, and the remaining two are determined to sum to 1
    freq_A1B1 = initial_freq_A1B1 / 100
    freq_A1B2 = initial_freq_A1B2 / 100
    freq_A2B1 = (1 - freq_A1B1 - freq_A1B2) / 2
    freq_A2B2 = (1 - freq_A1B1 - freq_A1B2) / 2
    
    # Calculate initial LD (D = freq(A1B1) * freq(A2B2) - freq(A1B2) * freq(A2B1))
    initial_D = freq_A1B1 * freq_A2B2 - freq_A1B2 * freq_A2B1
    
    # Recombination rate
    r = map_distance / 100
    
    # Simulate over generations
    max_gen = generations_to_show
    generations = list(range(max_gen + 1))
    
    haplotype_trajectories = {
        'A1B1': [freq_A1B1],
        'A1B2': [freq_A1B2],
        'A2B1': [freq_A2B1],
        'A2B2': [freq_A2B2]
    }
    
    D_values = [initial_D]
    
    # Allele frequencies (constant under random mating, no selection)
    p_A1 = freq_A1B1 + freq_A1B2
    p_B1 = freq_A1B1 + freq_A2B1
    
    current_freqs = [freq_A1B1, freq_A1B2, freq_A2B1, freq_A2B2]
    
    for gen in range(1, max_gen + 1):
        # Calculate D at this generation
        D = current_freqs[0] * current_freqs[3] - current_freqs[1] * current_freqs[2]
        D_values.append(D)
        
        # Update haplotype frequencies with recombination
        # After recombination: freq' = freq + r * (p_A * p_B - freq)
        new_A1B1 = current_freqs[0] - r * D
        new_A1B2 = current_freqs[1] + r * D
        new_A2B1 = current_freqs[2] + r * D
        new_A2B2 = current_freqs[3] - r * D
        
        current_freqs = [new_A1B1, new_A1B2, new_A2B1, new_A2B2]
        
        haplotype_trajectories['A1B1'].append(new_A1B1)
        haplotype_trajectories['A1B2'].append(new_A1B2)
        haplotype_trajectories['A2B1'].append(new_A2B1)
        haplotype_trajectories['A2B2'].append(new_A2B2)
    
    # Create visualization
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)
    
    # 1. Haplotype frequencies over time
    ax1 = fig.add_subplot(gs[0:2, :])
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    
    for (haplo, freqs), color in zip(haplotype_trajectories.items(), colors):
        ax1.plot(generations, freqs, 'o-', linewidth=2.5, markersize=6,
                label=haplo, color=color, alpha=0.8)
    
    ax1.set_xlabel('Generation', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Haplotype Frequency', fontsize=13, fontweight='bold')
    ax1.set_title(f'Haplotype Frequencies Over Time\nMap Distance: {map_distance} cM',
                 fontsize=15, fontweight='bold')
    ax1.legend(fontsize=11, loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-0.05, 1.05)
    
    # Add equilibrium frequencies as horizontal lines
    ax1.axhline(p_A1 * p_B1, color='#FF6B6B', linestyle='--', alpha=0.3, linewidth=1)
    ax1.axhline(p_A1 * (1-p_B1), color='#4ECDC4', linestyle='--', alpha=0.3, linewidth=1)
    
    # 2. D value decay
    ax2 = fig.add_subplot(gs[2, 0])
    ax2.plot(generations, D_values, 'o-', linewidth=2.5, markersize=6,
            color='purple', alpha=0.8)
    ax2.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Generation', fontsize=12, fontweight='bold')
    ax2.set_ylabel('D (Disequilibrium)', fontsize=12, fontweight='bold')
    ax2.set_title('LD Decay (D value)', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. Final state summary
    ax3 = fig.add_subplot(gs[2, 1])
    ax3.axis('off')
    
    final_freqs = {
        'A1B1': haplotype_trajectories['A1B1'][-1],
        'A1B2': haplotype_trajectories['A1B2'][-1],
        'A2B1': haplotype_trajectories['A2B1'][-1],
        'A2B2': haplotype_trajectories['A2B2'][-1]
    }
    
    expected_equilibrium = {
        'A1B1': p_A1 * p_B1,
        'A1B2': p_A1 * (1-p_B1),
        'A2B1': (1-p_A1) * p_B1,
        'A2B2': (1-p_A1) * (1-p_B1)
    }
    
    summary_text = f"""
    FINAL STATE (Gen {max_gen}):
    
    Haplotype  Current  Equilibrium
    ─────────────────────────────────
    A₁B₁       {final_freqs['A1B1']:.3f}    {expected_equilibrium['A1B1']:.3f}
    A₁B₂       {final_freqs['A1B2']:.3f}    {expected_equilibrium['A1B2']:.3f}
    A₂B₁       {final_freqs['A2B1']:.3f}    {expected_equilibrium['A2B1']:.3f}
    A₂B₂       {final_freqs['A2B2']:.3f}    {expected_equilibrium['A2B2']:.3f}
    
    D value:   {D_values[-1]:.6f}
    D initial: {initial_D:.6f}
    
    LD remaining: {(D_values[-1]/initial_D)*100:.1f}%
    """
    
    ax3.text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print("\n" + "="*70)
    print("INTERPRETATION:")
    print("="*70)
    
    if abs(D_values[-1]) < 0.001:
        print("✓ Population has reached LINKAGE EQUILIBRIUM")
        print("  Haplotype frequencies match expected values")
    else:
        print("⚠ Population still shows LINKAGE DISEQUILIBRIUM")
        print(f"  LD has decayed to {(D_values[-1]/initial_D)*100:.1f}% of initial value")
    
    if map_distance < 10:
        print("\n• Genes are TIGHTLY linked - LD persists")
    elif map_distance < 30:
        print("\n• Genes are MODERATELY linked - LD decays gradually")
    else:
        print("\n• Genes are LOOSELY linked or unlinked - LD decays rapidly")

# Interactive simulation
print("\n🧬 POPULATION GENETICS SIMULATOR")
print("Create a population and watch haplotype frequencies evolve!\n")

interact(simulate_population_ld,
         map_distance=FloatSlider(min=0.1, max=50, step=0.5, value=10,
                                 description='Map Distance (cM):',
                                 style={'description_width': 'initial'}),
         initial_freq_A1B1=FloatSlider(min=5, max=70, step=5, value=45,
                                      description='Initial A₁B₁ (%):',
                                      style={'description_width': 'initial'}),
         initial_freq_A1B2=FloatSlider(min=5, max=70, step=5, value=5,
                                      description='Initial A₁B₂ (%):',
                                      style={'description_width': 'initial'}),
         generations_to_show=IntSlider(min=5, max=100, step=5, value=50,
                                      description='Generations:',
                                      style={'description_width': 'initial'}));

---
## Part 5: Real-World Applications

Where do we see linkage and LD in actual genetics research?

In [ ]:
# Create comparison table
applications_data = {
    'Application': [
        'Gene Mapping',
        'GWAS Studies',
        'Evolutionary Studies',
        'Breeding Programs',
        'Disease Association',
        'Population History',
        'Forensics'
    ],
    'Uses Linkage?': [
        '✓✓✓ Primary',
        '✓ Indirect',
        '✓ Supporting',
        '✓✓✓ Primary',
        '✓ Supporting',
        '✗ Not used',
        '✓ Minor'
    ],
    'Uses LD?': [
        '✓ Supporting',
        '✓✓✓ Primary',
        '✓✓✓ Primary',
        '✓ Supporting',
        '✓✓✓ Primary',
        '✓✓✓ Primary',
        '✓✓ Important'
    ],
    'Example': [
        'Drosophila chromosome maps',
        'Finding disease SNPs',
        'Dating population splits',
        'Marker-assisted selection',
        'Type 2 diabetes risk loci',
        'Out-of-Africa migration',
        'Ancestry determination'
    ]
}

df_applications = pd.DataFrame(applications_data)

print("\n" + "="*80)
print("REAL-WORLD APPLICATIONS")
print("="*80)
print("\n")
print(df_applications.to_string(index=False))
print("\n" + "="*80)

# Detailed examples
print("\n📊 DETAILED EXAMPLES:\n")

print("1. GENE MAPPING (Classical Genetics)")
print("   Concept Used: LINKAGE")
print("   Example: Mapping Drosophila genes")
print("   • Cross flies with different traits")
print("   • Count recombinant offspring")
print("   • Calculate map distances")
print("   • Build chromosome maps\n")

print("2. GWAS - Genome-Wide Association Studies (Modern Genomics)")
print("   Concept Used: LINKAGE DISEQUILIBRIUM")
print("   Example: Finding Type 2 diabetes genes")
print("   • Genotype 10,000 people for 1 million SNPs")
print("   • Find SNPs associated with disease")
print("   • LD tells us the causal variant is nearby")
print("   • Don't need to genotype EVERY variant\n")

print("3. EVOLUTIONARY STUDIES (Population Genetics)")
print("   Concept Used: LINKAGE DISEQUILIBRIUM")
print("   Example: Human migration out of Africa")
print("   • African populations: Low LD (old, large populations)")
print("   • European/Asian populations: Higher LD (bottlenecks)")
print("   • LD patterns reveal population history")
print("   • Can date migration events\n")

print("4. PLANT BREEDING (Applied Genetics)")
print("   Concepts Used: LINKAGE + LD")
print("   Example: Breeding disease-resistant wheat")
print("   • Map resistance gene using linkage")
print("   • Find markers in LD with resistance")
print("   • Use markers to select plants (easier than testing disease)")
print("   • Speed up breeding programs\n")

print("="*80)

---
## Part 6: Common Misconceptions

Let's clear up some confusion!

In [ ]:
display(HTML("""
<style>
    .misconception {
        background-color: #ffcccc;
        padding: 15px;
        border-left: 5px solid #cc0000;
        margin: 10px 0;
        border-radius: 5px;
    }
    .truth {
        background-color: #ccffcc;
        padding: 15px;
        border-left: 5px solid #00cc00;
        margin: 10px 0;
        border-radius: 5px;
    }
    .myth-title {
        font-weight: bold;
        font-size: 16px;
        color: #cc0000;
    }
    .truth-title {
        font-weight: bold;
        font-size: 16px;
        color: #00aa00;
    }
</style>

<h2>🚫 MYTH vs ✓ REALITY</h2>

<div class="misconception">
    <span class="myth-title">❌ MYTH 1:</span> "Linkage and LD are the same thing"
</div>
<div class="truth">
    <span class="truth-title">✓ REALITY:</span> Linkage is about gene POSITION (fixed), LD is about allele ASSOCIATION (variable).
    Linkage is a chromosome map. LD is a population statistic.
</div>

<div class="misconception">
    <span class="myth-title">❌ MYTH 2:</span> "If genes are linked, they must show LD"
</div>
<div class="truth">
    <span class="truth-title">✓ REALITY:</span> Linked genes can show NO LD in old, randomly-mating populations.
    After many generations, even tightly linked genes reach equilibrium.
</div>

<div class="misconception">
    <span class="myth-title">❌ MYTH 3:</span> "LD only occurs between linked genes"
</div>
<div class="truth">
    <span class="truth-title">✓ REALITY:</span> LD can occur between genes on DIFFERENT chromosomes!
    Population admixture, selection, or drift can create LD anywhere.
</div>

<div class="misconception">
    <span class="myth-title">❌ MYTH 4:</span> "Recombination frequency and LD are the same"
</div>
<div class="truth">
    <span class="truth-title">✓ REALITY:</span> Recombination frequency measures linkage (fixed by genome).
    LD measures current allele associations (changes each generation).
</div>

<div class="misconception">
    <span class="myth-title">❌ MYTH 5:</span> "LD always decays at the same rate"
</div>
<div class="truth">
    <span class="truth-title">✓ REALITY:</span> LD decay rate depends on:<br>
    • Recombination rate (linkage affects this)<br>
    • Population size<br>
    • Selection<br>
    • Migration<br>
    • Mating patterns
</div>

<div class="misconception">
    <span class="myth-title">❌ MYTH 6:</span> "You can calculate linkage from LD"
</div>
<div class="truth">
    <span class="truth-title">✓ REALITY:</span> No! LD tells you HISTORY (what happened in this population).
    Linkage tells you GEOGRAPHY (where genes are on chromosomes).
    LD can suggest linkage, but doesn't measure it directly.
</div>
"""))

---
## Part 7: Quick Self-Assessment

Test your understanding!

In [ ]:
print("\n" + "="*80)
print("SELF-ASSESSMENT QUESTIONS")
print("="*80)
print("\nAnswer these questions to test your understanding:\n")

questions = [
    {
        'q': "Q1: Two genes are 15 cM apart. Is this describing linkage or LD?",
        'a': "LINKAGE (it's about physical distance)"
    },
    {
        'q': "Q2: In population X, allele A occurs with allele B 80% of the time (expected 50%). Is this linkage or LD?",
        'a': "LD (it's about allele associations in a population)"
    },
    {
        'q': "Q3: Can genes on different chromosomes show LD?",
        'a': "YES! (due to population structure, admixture, etc.)"
    },
    {
        'q': "Q4: If genes are tightly linked (1 cM), will LD persist longer than if they're 50 cM apart?",
        'a': "YES! (closer genes = slower LD decay)"
    },
    {
        'q': "Q5: In an old, randomly-mating population, two tightly linked genes (2 cM) show no LD. Is this possible?",
        'a': "YES! (after many generations, even linked genes reach equilibrium)"
    },
    {
        'q': "Q6: Which concept is more important for classical genetic mapping?",
        'a': "LINKAGE (we use recombination to map gene positions)"
    },
    {
        'q': "Q7: Which concept is more important for GWAS studies?",
        'a': "LD (we use it to find disease genes via association)"
    },
    {
        'q': "Q8: Does linkage change over time in a population?",
        'a': "NO! (it's fixed by the genome structure)"
    },
    {
        'q': "Q9: Does LD change over time in a population?",
        'a': "YES! (it decays with recombination each generation)"
    },
    {
        'q': "Q10: If D = 0, does that mean genes are not linked?",
        'a': "NO! It means no LD currently, but genes could still be physically linked"
    }
]

for i, item in enumerate(questions, 1):
    print(f"{item['q']}")
    print(f"   💡 Answer: {item['a']}\n")

print("="*80)

---
## Summary: The Complete Picture

### 🔑 Key Takeaways

1. **LINKAGE** = Physical proximity of genes (FIXED)
   - Measured in centiMorgans (cM)
   - Same for all individuals
   - Never changes
   - About chromosome structure

2. **LD** = Statistical association of alleles (VARIABLE)
   - Measured in D, D', r²
   - Population-specific
   - Changes over generations
   - About population history

3. **Relationship**: Linkage affects HOW FAST LD decays
   - Tight linkage → Slow LD decay
   - No linkage → Fast LD decay
   - But LD can exist without linkage!

4. **Applications**:
   - Classical mapping → Use linkage
   - GWAS → Use LD
   - Evolution studies → Use LD
   - Breeding → Use both!

---

**Remember:** 
- Linkage is about WHERE genes are
- LD is about HOW alleles associate in populations
- They're related but distinct concepts!

**Congratulations!** You now understand the difference between linkage and linkage disequilibrium! 🎉
